# The Perceptron

---

## Overview

The **Perceptron** is the simplest neural network model. a single neuron with a step activation function. It is a binary classifier for *linearly separable* data.

Given a feature vector $\mathbf{x}$, the perceptron computes:

$$z = \mathbf{w} \cdot \mathbf{x} + b$$

$$\hat{y} = \begin{cases} +1 & \text{if } z \geq 0 \\ -1 & \text{if } z < 0 \end{cases}$$

---

## Update Rule

For each misclassified sample $(\mathbf{x}^{(i)}, y^{(i)})$:

$$\mathbf{w} \leftarrow \mathbf{w} - \eta (\hat{y}^{(i)} - y^{(i)}) \mathbf{x}^{(i)}$$
$$b \leftarrow b - \eta (\hat{y}^{(i)} - y^{(i)})$$

where $\eta$ is the learning rate.

---

**Dataset:** Heart Disease (`heart.csv`)  
**Task:** Binary classification. predict presence of heart disease (labels: $-1$, $+1$).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme()

from rice_ml.supervised_learning import Perceptron
from rice_ml.preprocess import StandardScaler, train_test_split
from rice_ml.metrics import accuracy_score, confusion_matrix

In [ ]:
try:
    df = pd.read_csv('../../../data/heart.csv')
    target_col = 'target'
    X = df.drop(columns=[target_col]).values.astype(float)
    y_raw = df[target_col].values
    print(f'Loaded heart dataset: {X.shape}')
except FileNotFoundError:
    from sklearn.datasets import load_breast_cancer
    data = load_breast_cancer()
    X, y_raw = data.data, data.target
    print('CSV not found. using breast cancer dataset')

# Perceptron uses -1 / +1 labels
y = np.where(y_raw == 0, -1, 1)
print(f'Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}')

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

## Train the Perceptron

The model iterates over all training samples each epoch, updating weights only for misclassified points.

In [ ]:
clf = Perceptron(eta=0.1, epochs=100)
clf.fit(X_train, y_train)

print(f'Epochs run:     {len(clf.errors_)}')
print(f'Final errors:   {clf.errors_[-1] if clf.errors_ else 0}')

In [ ]:
# Plot misclassification count per epoch
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(clf.errors_) + 1), clf.errors_, marker='o', color='steelblue')
plt.xlabel('Epoch', fontsize=15)
plt.ylabel('Misclassifications', fontsize=15)
plt.title('Perceptron: Misclassifications per Epoch', fontsize=18)
plt.show()

In [ ]:
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {acc:.4f}')

cm = confusion_matrix(y_test, y_pred, labels=[-1, 1])
print(f'Confusion Matrix:\n{cm}')

In [ ]:
# Confusion matrix heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['-1', '+1'], yticklabels=['-1', '+1'])
plt.xlabel('Predicted', fontsize=13)
plt.ylabel('True', fontsize=13)
plt.title('Perceptron: Confusion Matrix', fontsize=16)
plt.show()

## Interpretation

- The perceptron converges when `errors == 0`, meaning all training points are correctly classified.
- If the data is **not linearly separable**, the error curve will not reach zero. logistic regression handles this case better.
- The weight vector $\mathbf{w}$ defines the separating hyperplane: $\mathbf{w} \cdot \mathbf{x} + b = 0$.